# 02 — Data preparation + SFT training (Task 3)

Accelerator: **GPU T4 x2**, Internet **on**. Expected time: ~10 min data prep + ~1 h training.

1. Builds the fixed training subsets (SFT 30k/1k, HH-RLHF 15k/1k, UltraFeedback 15k/1k).
2. Trains SFT on both GPUs with DDP (`torchrun`): per-device batch 2 x accumulation 8 x 2 GPUs = effective batch 32.
3. Sanity-generates 3 prompts from the best checkpoint.

Outputs: `outputs/sft/` (`metrics.jsonl`, `config.json`, `summary.json`, `checkpoints/best`, `checkpoints/final`)
and `data/manifests/`. When it finishes: **Save Version** — the DPO notebook attaches this notebook's output.

In [ ]:
REPO_URL = "https://github.com/madgod2004/small-model-alignment-starter.git"
REPO_DIR = "/kaggle/working/small-model-alignment-starter"
BRANCH = "main"

In [ ]:
import os
if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors matplotlib "peft>=0.11.0" "trl>=0.12.0" pytest
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi
!python scripts/check_env.py --expect_gpus 2

## 1. Data preparation

In [ ]:
!python scripts/prepare_data.py --tasks sft hh uf 2>&1 | grep -v Warning

In [ ]:
import json
for name in ["sft", "hh", "uf"]:
    manifest = json.load(open(f"data/manifests/{name}.json"))
    for split, info in manifest["splits"].items():
        print(f"{name}/{split}: {info['stats']}")

## 2. SFT training (DDP on 2 GPUs)
Progress prints every 10 optimizer steps; validation every 100. `PYTHONUNBUFFERED=1` makes the log stream live.
For a single GPU instead: `python scripts/train_sft.py --config configs/sft.yaml` (accumulation becomes 16).

In [ ]:
!mkdir -p outputs/logs
!PYTHONUNBUFFERED=1 torchrun --nproc_per_node 2 scripts/train_sft.py --config configs/sft.yaml 2>&1 | grep -v -i warn | tee outputs/logs/train_sft.log

In [ ]:
import json
import matplotlib.pyplot as plt

rows = [json.loads(line) for line in open("outputs/sft/metrics.jsonl")]
train = [r for r in rows if r["type"] == "train"]
evals = [r for r in rows if r["type"] == "eval"]
print(json.dumps(json.load(open("outputs/sft/summary.json")), indent=1))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([r["step"] for r in train], [r["train_loss"] for r in train], label="train loss (per logged step)", alpha=0.6)
ax.plot([r["step"] for r in evals], [r["val_loss"] for r in evals], "o-", label="validation loss")
ax.set_xlabel("optimizer step")
ax.set_ylabel("response-token NLL")
ax.set_title("SFT on Qwen2.5-0.5B")
ax.legend()
plt.show()

## 3. Sanity check: 3 prompts from the best SFT checkpoint
Check the format, that responses end (EOS), and length.

In [ ]:
!python scripts/generate.py --model_path outputs/sft/checkpoints/best --generator qwen2.5-0.5b-sft --eval_set eval_sets/alpaca_eval_300_seed42.json --output_path outputs/generations/sanity_sft.json --limit 3 --print --overwrite 2>&1 | grep -v -i warn

## 4. Archive the small artifacts
Checkpoints (~2 GB each) stay in the saved notebook output; the zip holds metrics, configs, manifests, and logs.

In [ ]:
!rm -f outputs/generations/sanity_sft*.json
!zip -qr /kaggle/working/sft_run_outputs.zip outputs/sft/metrics.jsonl outputs/sft/config.json outputs/sft/summary.json outputs/sft/checkpoints/best/training_state.json data/manifests outputs/logs/train_sft.log
!ls -lh /kaggle/working/*.zip && du -sh outputs/sft/checkpoints/*